# Pandas

This module introduces pandas, the in-memory tabular data library that
underpins most data work in Python. Readers are assumed to know Python and
to have at least seen a SQL table, an R `data.frame`, or an Excel pivot.
The focus here is on the pandas data model (labeled axes, vectorized
execution, the index as a first-class object) and on the idioms that make
the library efficient rather than fluent translations from other tools.

The topics below are arranged linearly for review. Structural grouping
(sections, chapters) can be applied later. The same running example, a
small sales-plan DataFrame, is reused throughout, so each topic introduces
exactly one new idea.

https://www.datacamp.com/tutorial/pandas-read-csv

---

## Topic list

1. Pandas in the data tool landscape
2. The running example
3. Series and DataFrame
4. The index is everything
5. Construction and basic I/O
6. Selecting rows and columns
7. Boolean indexing
8. Vectorization vs. iteration
9. Missing values
10. dtypes and memory
11. Adding, dropping, renaming
12. Sorting
13. Aggregations
14. groupby
15. Reshaping: pivot, melt, stack, unstack
16. MultiIndex
17. Combining: concat, merge, join
18. Time series essentials
19. String operations
20. Categorical data
21. Method chaining
22. Copy vs view, SettingWithCopyWarning
23. When pandas is the wrong tool
24. Real-world design principles
25. Common mistakes

---

## 1. Pandas in the data tool landscape

Pandas is a tabular data library built on top of NumPy. It was created by
Wes McKinney at AQR Capital in 2008, released publicly in 2009, and has
been the de facto standard for in-memory tabular work in Python since
around 2012. Its closest conceptual relatives are R's `data.frame` and
SQL's relational table; its closest implementation relative is NumPy,
whose ndarray backs every pandas column.

Two design decisions shape almost everything that follows. First, pandas
data is **labeled**: rows and columns carry an index, and operations align
on those labels rather than on row position. Second, pandas inherits
NumPy's vectorized execution model: most methods operate on whole columns
at once, and explicit Python-level loops are almost always the wrong tool.

Pandas is at its best when the dataset fits comfortably in memory (rough
ceiling: a few gigabytes, tens of millions of rows), the work is
interactive or exploratory, the schema is heterogeneous (mixed dtypes,
missing values, dates, categoricals), and the output is another DataFrame,
a chart, or a small report. It is the wrong choice for streaming
pipelines, distributed execution, or queries over hundreds of millions of
rows. Topic 23 lists alternatives.

For tm1py work specifically, pandas is the lingua franca: `execute_mdx`
and view readers can return DataFrames directly, write paths accept them,
and most analytical work between read and write is most naturally
expressed in pandas.

## 2. The running example

The examples in this module use a small sales plan loaded from a flat
table. Real planning data has more columns and many more rows; the shape
is what matters.

In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "period":  ["2026Q1", "2026Q1", "2026Q1", "2026Q2", "2026Q2", "2026Q2"],
    "region":  ["Europe", "Americas", "Asia", "Europe", "Americas", "Asia"],
    "product": ["Standard", "Standard", "Standard", "Premium", "Premium", "Premium"],
    "revenue": [120_000.0, 250_000.0, 90_000.0, 135_000.0, 270_000.0, 110_000.0],
    "units":   [1200, 2500, 900, 540, 1080, 440],
})

sales
#    period    region   product   revenue  units
# 0  2026Q1    Europe   Standard  120000.0   1200
# 1  2026Q1    Americas Standard  250000.0   2500
# 2  2026Q1    Asia     Standard   90000.0    900
# 3  2026Q2    Europe   Premium   135000.0    540
# 4  2026Q2    Americas Premium   270000.0   1080
# 5  2026Q2    Asia     Premium   110000.0    440

`period`, `region`, and `product` are the dimensional columns; `revenue`
and `units` are the measures. This shape (a few key columns plus numeric
measures) is the canonical input for pivots, group reductions, and
eventual write back to a TM1 cube.

## 3. Series and DataFrame

Pandas exposes two core data structures. A `Series` is a labeled
one-dimensional array. A `DataFrame` is a labeled two-dimensional table
whose columns are Series sharing a common row index.

In [ ]:
revenue: pd.Series = sales["revenue"]
type(revenue)            # <class 'pandas.core.series.Series'>
revenue.name             # 'revenue'
revenue.index            # RangeIndex(start=0, stop=6, step=1)
revenue.values           # numpy array of the underlying data
revenue.dtype            # float64

type(sales)              # <class 'pandas.core.frame.DataFrame'>
sales.shape              # (6, 5)
sales.columns            # Index(['period', 'region', 'product', 'revenue', 'units'], ...)
sales.index              # RangeIndex(start=0, stop=6, step=1)
sales.dtypes             # one dtype per column

Most DataFrame methods are also available on Series, and selecting a
single column from a DataFrame returns a Series. Selecting two or more
columns returns a DataFrame, even if the list contains only one name:
`sales[["revenue"]]` is a DataFrame, `sales["revenue"]` is a Series. This
single-name vs. list-of-names distinction is the most common source of
"why does my output have a different shape than I expected".

## 4. The index is everything

Every Series and DataFrame has an index. By default it is a `RangeIndex`
(0, 1, 2, ...), but any column with unique or even non-unique labels can
become the index. The index is not cosmetic. Every operation that
combines data (arithmetic, joins, alignment) aligns on the index, not on
row order.

In [ ]:
revenue = sales.set_index(["period", "region"])["revenue"]
revenue.loc["2026Q1", "Europe"]    # 120000.0
revenue.loc["2026Q1"]              # all regions in 2026Q1, as a Series

a = pd.Series([10, 20, 30], index=["x", "y", "z"])
b = pd.Series([1, 2, 3], index=["y", "z", "w"])

a + b
# w     NaN
# x     NaN
# y    21.0
# z    32.0
# dtype: float64

Two values are added only when their labels match; mismatches produce
`NaN`. This is the behavior that surprises newcomers from SQL, where rows
combine by position or join key. In pandas the index is _always_ the
implicit join key.

A practical consequence: when an operation produces unexpected `NaN`
values or a result with the wrong number of rows, suspect index
misalignment first. `df.reset_index(drop=True)` is the usual fix when row
position is what matters and the labels are not meaningful.

## 5. Construction and basic I/O

DataFrames are most often constructed by reading a file or by passing a
dict of columns. The dict-of-columns form is the canonical way to build
test data and the most readable way to write small examples.

In [ ]:
from pathlib import Path

sales = pd.read_csv(Path("data/sales.csv"))
sales = pd.read_excel("data/sales.xlsx", sheet_name="Plan")
sales = pd.read_parquet("data/sales.parquet")

sales.to_csv("out/sales.csv", index=False)
sales.to_parquet("out/sales.parquet")

A few flags are worth knowing on first contact. `index=False` on `to_csv`
prevents the index from being written as an unnamed leading column, which
is almost always what is wanted for interchange files. `parse_dates=` on
`read_csv` converts string date columns at read time, which is much
faster than calling `pd.to_datetime` afterward. `dtype=` lets the reader
fix dtypes up front rather than re-coercing later.

For tm1py work, the same DataFrame can be obtained from a TM1 view via
`tm1.cubes.cells.execute_view_dataframe(...)`. The shape and dtypes are
controllable through the call and through subsequent pandas operations,
not through file format flags. Topic 16 returns to this point.

## 6. Selecting rows and columns

Pandas offers three selection styles, and confusion between them is
responsible for a large share of beginner bugs.

In [ ]:
sales["revenue"]              # column by name, returns Series
sales[["revenue", "units"]]   # multiple columns, returns DataFrame
sales[0:3]                    # row slice by position (rare; ambiguous)

sales.loc[0, "revenue"]       # label-based: row label 0, column 'revenue'
sales.loc[:, "revenue"]       # all rows, one column
sales.loc[sales["region"] == "Europe", ["revenue", "units"]]

sales.iloc[0, 3]              # position-based: row 0, column 3
sales.iloc[0:3, :]            # first three rows, all columns
sales.iloc[-1]                # last row

The rule of thumb is: use `.loc` when working with labels (the typical
case once an index has been set), `.iloc` when working with positions
(rare in production code; common when porting numpy patterns), and the
plain bracket `df[...]` only for column selection. Plain bracket
selection of rows by integer slice exists for historical reasons and is
best avoided once an index has been set, because the slice meaning shifts
with the index type.

A read-only single-cell access has a faster pair: `.at` and `.iat`.
`sales.at[0, "revenue"]` is identical in meaning to `sales.loc[0,
"revenue"]` but skips the broader `.loc` machinery.

## 7. Boolean indexing

A Series of booleans, the same length as the DataFrame, selects rows
where the value is `True`. This is the standard pandas filter idiom.

In [ ]:
sales[sales["region"] == "Europe"]
sales[sales["revenue"] > 100_000]

mask = (sales["region"] == "Europe") & (sales["revenue"] > 100_000)
sales[mask]

sales[sales["region"].isin(["Europe", "Asia"])]
sales[~sales["region"].isin(["Europe", "Asia"])]   # ~ is "not"

Two operator details trip up newcomers. First, `&`, `|`, and `~` are the
elementwise boolean operators; the Python keywords `and`, `or`, and `not`
do not work on Series and raise an error or give silently wrong results.
Second, the operator precedence of `&` is _higher_ than that of `==`, so
parentheses around each condition are mandatory: `(a == 1) & (b == 2)`,
never `a == 1 & b == 2`.

For "any of these values", `.isin([...])` is faster, more readable, and
works on Series of any dtype. For range checks, `.between(low, high)`
returns the same shape as the comparison operators.

## 8. Vectorization vs. iteration

The single largest performance difference between idiomatic pandas and
naive pandas is whether the code iterates row by row or expresses the
work as operations on whole columns. Vectorized operations push the loop
into NumPy's C implementation; row iteration runs the loop in Python.

In [ ]:
# Slow: explicit row iteration
totals = []
for _, row in sales.iterrows():
    totals.append(row["revenue"] - row["units"] * 50)
sales["margin"] = totals

# Fast: vectorized arithmetic
sales["margin"] = sales["revenue"] - sales["units"] * 50

`apply` sits between the two. It accepts a function and runs it once per
row or column, which is faster than `iterrows` but still slower than
vectorized arithmetic by a factor of 10 to 100. `apply` is appropriate
when the per-row logic genuinely cannot be vectorized (calling an external
service, complex branching that no NumPy expression captures); it is not
a default. When in doubt, write the vectorized form first and reach for
`apply` only after profiling.

For string operations, comparisons, and arithmetic, vectorized forms are
always available. They are listed in Topic 19 (`.str` accessor) and
Topic 13 (aggregations).

## 9. Missing values

Pandas represents missing values as `NaN` (a float) in numeric columns,
`NaT` (not a time) in datetime columns, and `None` or `pd.NA` in object
and the newer nullable dtypes. `NaN` is contagious in arithmetic: any
operation involving `NaN` produces `NaN`.

In [ ]:
sales.loc[2, "revenue"] = float("nan")

sales["revenue"].isna()         # boolean Series, True where missing
sales["revenue"].notna()
sales.dropna(subset=["revenue"])         # drop rows where revenue is NaN
sales["revenue"].fillna(0)               # replace missing with 0
sales["revenue"].fillna(sales["revenue"].mean())

sales["revenue"].sum()           # NaN entries are skipped by default
sales["revenue"].sum(skipna=False)

Two pitfalls are worth flagging. First, comparing to `NaN` always returns
`False`: `sales["revenue"] == float("nan")` selects nothing. Use
`.isna()` instead. Second, integer columns with missing values are
silently promoted to `float64` in classic pandas, because `NaN` is not a
valid integer. The newer nullable integer dtype `Int64` (capital I)
preserves integer semantics and stores missing values as `pd.NA`. For
data that originates from TM1, where missing cells are common and
distinguishable from zero, the nullable types are usually the better
choice.

## 10. dtypes and memory

Each column has a single dtype. The common ones are `int64`, `float64`,
`bool`, `object` (a catch-all for Python objects, typically strings),
`datetime64[ns]`, `category`, and the nullable variants `Int64`,
`Float64`, `boolean`, `string`.

In [ ]:
sales.dtypes
# period      object
# region      object
# product     object
# revenue     float64
# units       int64

sales.memory_usage(deep=True)    # per-column memory, including object payloads

sales = sales.astype({
    "period":  "category",
    "region":  "category",
    "product": "category",
    "units":   "Int64",
})

`object` columns store Python objects and pay the full Python overhead per
value (roughly 50 bytes per short string plus the string itself).
Converting low-cardinality string columns to `category` typically reduces
their footprint by an order of magnitude and speeds up `groupby` and
equality comparisons. For columns that originate as TM1 dimension
elements, `category` is almost always the right dtype: the cardinality is
bounded by the dimension size, and the same elements repeat across many
rows.

## 11. Adding, dropping, renaming

Column-level edits are common enough that pandas offers four overlapping
ways to do them. The choice is usually about whether the result is a new
DataFrame or a mutation of the existing one.

In [ ]:
sales["margin"] = sales["revenue"] - sales["units"] * 50    # in place add
sales = sales.assign(margin=lambda d: d["revenue"] - d["units"] * 50)  # returns new

sales = sales.drop(columns=["margin"])
sales = sales.rename(columns={"region": "geography"})

sales.columns = ["period", "geography", "product", "revenue", "units"]   # full rename

`assign` is preferred inside method chains (Topic 21) because it returns a
new DataFrame; the bracket form mutates and breaks chains. `drop` and
`rename` both accept `inplace=True`, but inplace operations return `None`
and are best avoided: they preclude chaining and provide no measurable
performance benefit in modern pandas.

## 12. Sorting

`sort_values` orders rows by one or more columns; `sort_index` orders
them by the index. Both are stable and both default to ascending order.

In [ ]:
sales.sort_values("revenue", ascending=False)
sales.sort_values(["region", "revenue"], ascending=[True, False])

sales.set_index(["period", "region"]).sort_index()

A sorted index unlocks faster slice access (`df.loc["2026Q1":"2026Q3"]`)
and is required for some operations on time series. After any operation
that scrambles index order, a `sort_index()` call is cheap insurance.

## 13. Aggregations

Reductions collapse a Series or DataFrame to a smaller shape, typically a
scalar (for a Series) or a Series (for a DataFrame). The standard ones
are `sum`, `mean`, `min`, `max`, `count`, `nunique`, `std`, `var`,
`median`, `quantile`.

In [ ]:
sales["revenue"].sum()                  # scalar
sales["revenue"].mean()
sales[["revenue", "units"]].sum()       # Series, one entry per column
sales.sum(numeric_only=True)            # all numeric columns

sales.agg({
    "revenue": ["sum", "mean"],
    "units":   "sum",
})
#         revenue          units
# sum     975000.0        6660.0
# mean    162500.0           NaN

`agg` accepts a dict mapping column names to functions or lists of
functions, and produces a DataFrame indexed by the function names. The
companion method `describe()` returns a quick statistical summary
(`count`, `mean`, `std`, `min`, quartiles, `max`) for each numeric column;
useful at the start of any analysis.

## 14. groupby

`groupby` implements the split-apply-combine model: split the DataFrame
into groups, apply a function to each group, combine the results back
into a single output. It is the most heavily used operation in pandas
after column selection.

In [ ]:
sales.groupby("region")["revenue"].sum()
# region
# Americas    520000.0
# Asia        200000.0
# Europe      255000.0

sales.groupby(["period", "region"])[["revenue", "units"]].sum()

sales.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    avg_units=("units", "mean"),
    n_rows=("revenue", "size"),
)

The named-aggregation form (`new_name=("column", function)`) is the
clearest in production code: it specifies output column names explicitly
rather than relying on multi-level column headers.

Three groupby variants beyond aggregation are worth knowing. `transform`
returns a result with the same shape as the input, useful for adding
group-level statistics back as columns
(`sales["region_total"] = sales.groupby("region")["revenue"].transform("sum")`).
`filter` keeps or drops whole groups based on a predicate. `apply`
accepts an arbitrary function that returns a DataFrame, Series, or
scalar; it is the most flexible and the slowest, with the same caveats as
the row-wise `apply` in Topic 8.

## 15. Reshaping: pivot, melt, stack, unstack

Tabular data exists in two canonical shapes. **Long** form has one row
per observation and a column for each attribute, including the measure
identifier. **Wide** form spreads one or more attributes across columns.
Pandas converts between them with `melt` (wide to long), `pivot` and
`pivot_table` (long to wide), and the index-aware pair `stack` and
`unstack`.

In [ ]:
wide = sales.pivot_table(
    index=["period", "product"],
    columns="region",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)
# region                Americas      Asia    Europe
# period product
# 2026Q1 Standard       250000.0   90000.0  120000.0
# 2026Q2 Premium        270000.0  110000.0  135000.0

long = wide.stack().reset_index(name="revenue")

Long form is the right shape for storage, transfer, and most aggregation;
wide form is the right shape for human reading and for write back to a
TM1 cube view that has region (or any other dimension) on columns.
`pivot_table` handles duplicates by aggregating; `pivot` requires unique
index/column pairs and raises otherwise.

## 16. MultiIndex

When `set_index` or `groupby` is called with multiple columns, the
resulting object has a `MultiIndex`: an index whose labels are tuples.
This is the natural pandas representation of multi-dimensional data and
maps directly onto a TM1 cellset, where each cell is identified by a
tuple of dimension elements.

In [ ]:
plan = sales.set_index(["period", "region", "product"])

plan.loc[("2026Q1", "Europe", "Standard"), "revenue"]   # 120000.0
plan.loc["2026Q1"]                                       # all rows in 2026Q1
plan.loc[(slice(None), "Europe"), :]                     # all periods, Europe only
plan.xs("Europe", level="region")                        # cleaner than slice(None)

A MultiIndex on the columns axis is also possible and often appears as
the output of `pivot_table` or `groupby(...).agg([...])`. `df.columns`
on such a DataFrame is itself a `MultiIndex`. To flatten it to plain
column names, join the levels: `df.columns = ["_".join(c) for c in
df.columns]`.

When reading from tm1py, a cellset DataFrame typically arrives with a
MultiIndex over the dimensions and a single `Value` column. Most
analytical work then operates on this shape directly: aggregation is
`groupby(level=...)`, slicing is `xs`, and write back constructs the
same MultiIndex shape and hands it to `cells.write`.

## 17. Combining: concat, merge, join

Three operations combine DataFrames. `concat` stacks them along an axis.
`merge` performs a SQL-style join on columns or indices. `join` is a
convenience wrapper around `merge` that defaults to joining on the
index.

In [ ]:
q1 = sales[sales["period"] == "2026Q1"]
q2 = sales[sales["period"] == "2026Q2"]
both = pd.concat([q1, q2], ignore_index=True)

prices = pd.DataFrame({
    "product": ["Standard", "Premium"],
    "list_price": [100.0, 250.0],
})

joined = sales.merge(prices, on="product", how="left")

`how=` controls join semantics: `"inner"`, `"left"`, `"right"`, `"outer"`.
The default is `"inner"`, which is rarely what is wanted in a planning
context where preserving the left side (the fact table) is usually the
intent. Always specify `how=` explicitly. After a left join, check for
unintended `NaN` values in the joined columns; they indicate keys present
on the left but missing from the right.

`indicator=True` adds a `_merge` column showing where each row came from
(`left_only`, `right_only`, `both`). It is the fastest way to debug a
join that returns the wrong number of rows.

## 18. Time series essentials

Pandas treats datetime columns as first-class data and provides a
substantial library of time aware operations. The minimum useful subset
is small.

In [ ]:
sales["period_start"] = pd.to_datetime(
    sales["period"].str.replace("Q1", "-01-01")
                   .str.replace("Q2", "-04-01")
                   .str.replace("Q3", "-07-01")
                   .str.replace("Q4", "-10-01")
)

sales.set_index("period_start").resample("Q")["revenue"].sum()

dates = pd.date_range("2026-01-01", periods=12, freq="MS")  # month starts

`resample` is the time-series analog of `groupby`: it buckets rows by a
time frequency (`"D"`, `"W"`, `"M"`, `"Q"`, `"Y"`, plus offsets like
`"MS"` for month start) and then aggregates. It requires a
`DatetimeIndex`. For TM1 planning data, where periods are typically
already discrete (months, quarters, years), `resample` is less useful
than plain `groupby` on the period column; reach for it when working with
genuine timestamped events (transactions, log entries) rather than
prebucketed plan data.

## 19. String operations

The `.str` accessor exposes vectorized string methods. They are the
correct tool for any column-wide string transformation; calling Python's
`str.upper` inside an `apply` is slower and less idiomatic.

In [ ]:
sales["region"].str.upper()
sales["region"].str.startswith("E")
sales["product"].str.contains("prem", case=False)
sales["product"].str.replace("Premium", "Pro")
sales["period"].str.split("Q", expand=True)
#         0  1
# 0    2026  1
# 1    2026  1
# ...

`expand=True` on splitting methods produces a DataFrame, one column per
piece. This is the idiomatic way to split a compound key column (a
typical pattern when reading TM1 dimension element names that encode
multiple attributes).

## 20. Categorical data

A categorical column stores its values as integer codes plus a list of
unique categories. For low-cardinality string columns the memory savings
are substantial, equality comparisons are faster, and an explicit
ordering can be attached.

In [ ]:
sales["region"] = sales["region"].astype("category")
sales["region"].cat.categories       # Index(['Americas', 'Asia', 'Europe'], ...)
sales["region"].cat.codes            # 0, 1, 2 ...

regions_ordered = pd.CategoricalDtype(
    categories=["Europe", "Americas", "Asia"],
    ordered=True,
)
sales["region"] = sales["region"].astype(regions_ordered)
sales.sort_values("region")          # uses the explicit ordering

For TM1 work the bridge is direct: a dimension is, by construction, a
fixed set of element names, often in a specific order. Mapping it to a
pandas `CategoricalDtype` preserves that information and makes
downstream merges and groupbys both faster and safer (a typo in a region
name produces `NaN` rather than a silent new category).

## 21. Method chaining

A pandas pipeline reads more clearly as a chain of method calls than as a
sequence of intermediate variable assignments. Each call returns a new
DataFrame, so chains compose without mutating their input.

In [ ]:
summary = (
    sales
    .assign(margin=lambda d: d["revenue"] - d["units"] * 50)
    .query("region != 'Asia'")
    .groupby(["period", "region"], as_index=False)
    .agg(total_revenue=("revenue", "sum"),
         total_margin=("margin", "sum"))
    .sort_values(["period", "total_revenue"], ascending=[True, False])
)

A few methods exist specifically to keep chains intact. `.assign` adds
columns and returns a new DataFrame. `.query("col == value")` filters by
a string expression (less powerful than boolean indexing but more
chainable). `.pipe(func, *args)` inserts an arbitrary function into the
chain: `df.pipe(my_function, threshold=100)` is equivalent to
`my_function(df, threshold=100)` but reads in flow order.

The trade-off is that intermediate values are not held in named
variables, which makes debugging a long chain harder. The usual
mitigation is to break a chain at a logical seam, name the partial
result, and continue.

## 22. Copy vs view, SettingWithCopyWarning

When a selection returns a _view_ into the original DataFrame, assigning
into it modifies the original. When it returns a _copy_, assignment
modifies only the copy. Pandas does not document which is which for every
operation, and the difference is the source of the infamous
`SettingWithCopyWarning`.

In [ ]:
europe = sales[sales["region"] == "Europe"]
europe["revenue"] = 0          # SettingWithCopyWarning
# Did this modify sales? Maybe. The contract is not stable.

The reliable fix is to make the intent explicit. If the goal is a new
independent DataFrame, call `.copy()` immediately:

In [ ]:
europe = sales[sales["region"] == "Europe"].copy()
europe["revenue"] = 0          # safe, sales is untouched

If the goal is to modify the original, address it directly with `.loc`:

In [ ]:
sales.loc[sales["region"] == "Europe", "revenue"] = 0

Pandas 3.x is moving toward "Copy-on-Write" semantics, in which the
warning becomes an error and chained assignment is unambiguous. Until
that is the default in your environment, treat the warning as a bug
report from the library: it is telling you that the operation's effect
is undefined.

## 23. When pandas is the wrong tool

Pandas is single threaded, holds all data in memory, and pays Python
overhead at every method call. For workloads outside its sweet spot,
better tools exist.

- **Polars** is a Rust-based DataFrame library with similar API surface,
  multi-threaded execution, and a query optimizer. For pure data
  transformation on tens of millions of rows, polars is faster by an
  order of magnitude with similar code.
- **DuckDB** runs SQL over Parquet or pandas DataFrames in process. It
  is the right tool for analytical queries that read like SQL,
  especially joins and window functions over large data.
- **Dask** distributes pandas-like operations across multiple processes
  or machines. It is the natural choice when the dataset does not fit on
  one machine but the operations remain DataFrame shaped.
- **NumPy directly** is the right tool when the data is purely numeric,
  the index does not matter, and the bottleneck is arithmetic. The
  overhead of going through pandas is real.

For tm1py work the dataset is bounded by the cube and is almost always
small enough that pandas is the right answer. Awareness of the
alternatives matters mostly for upstream and downstream stages of the
pipeline, where source files or output destinations may be much larger.

## 24. Real-world design principles

Once the mechanics are clear, the practical question is when and how to
use them. A handful of guidelines apply to most situations.

**Vectorize by default; reach for `apply` only with a reason.** A
vectorized expression is shorter, faster, and clearer than the loop or
`apply` form. The 10x to 100x speedup is real and matters at the row
counts where pandas is normally used. Topic 8 covered the mechanics; the
discipline is to write the vectorized form first and keep it unless the
logic genuinely demands per-row Python.

**Set the index deliberately.** A meaningful index unlocks `.loc`
selection, alignment in arithmetic, fast joins, and `xs` slicing for
multi-level data. A meaningless `RangeIndex` works fine for flat tables
but is wasted on dimensional data. For TM1 cellsets in particular, the
right index is almost always a MultiIndex over the dimensions, and the
work after that is much easier.

**Keep dtypes tight.** `object` columns are the slow path. Convert
low-cardinality strings to `category`, enable nullable integer dtypes
where missing values are meaningful, and parse dates at read time. A
brief `df.dtypes` audit at each stage of a pipeline catches dtype drift
before it becomes a performance problem.

**Prefer chains over intermediate variables, but break them at seams.**
A method chain reads as a pipeline; a sequence of `df1`, `df2`, `df3`
assignments reads as a sequence of unrelated mutations. Long chains
become hard to debug, so split them at logical boundaries (after a
groupby, after a join) and name the intermediate.

**Treat `SettingWithCopyWarning` as an error.** The warning marks an
operation whose effect is not well defined. Add `.copy()` or address the
target through `.loc`; do not silence the warning.

**Validate at boundaries, trust internally.** Once a DataFrame has been
read, normalized, dtype-checked, and indexed, downstream code can trust
its shape. Repeated defensive checks at every step add noise; one
deliberate validation step right after ingestion is worth a dozen
ad hoc `assert` statements scattered later.

## 25. Common mistakes

A short collection of errors that are easy to make and worth recognizing
early.

**Using Python `and`/`or` in boolean masks.** These keywords do not
broadcast over Series; the operators `&`, `|`, and `~` do.

In [ ]:
# Wrong
sales[sales["region"] == "Europe" and sales["revenue"] > 100_000]
# ValueError: The truth value of a Series is ambiguous

# Correct
sales[(sales["region"] == "Europe") & (sales["revenue"] > 100_000)]

**Comparing to NaN with `==`.** `NaN` is not equal to itself, so this
test always returns `False`.

In [ ]:
# Wrong
sales[sales["revenue"] == float("nan")]   # selects nothing

# Correct
sales[sales["revenue"].isna()]

**Iterating with `iterrows` to compute a new column.** This is the
canonical slow pattern. Vectorize.

In [ ]:
# Wrong
margins = []
for _, row in sales.iterrows():
    margins.append(row["revenue"] - row["units"] * 50)
sales["margin"] = margins

# Correct
sales["margin"] = sales["revenue"] - sales["units"] * 50

**Chained assignment.** Modifying a DataFrame through a chained selection
is not guaranteed to affect the original.

In [ ]:
# Wrong: may or may not modify sales, raises SettingWithCopyWarning
sales[sales["region"] == "Europe"]["revenue"] = 0

# Correct
sales.loc[sales["region"] == "Europe", "revenue"] = 0

**Default `merge` how.** The default join is inner, which silently drops
rows whose key has no match on the other side. Specify `how=` explicitly
on every merge.

In [ ]:
# Risky: drops unmatched rows without warning
sales.merge(prices, on="product")

# Correct
sales.merge(prices, on="product", how="left")

**Forgetting `index=False` on `to_csv`.** Writing a CSV with the default
settings includes the index as an unnamed leading column, which round
trips badly when the file is read back.

In [ ]:
# Wrong: produces an extra unnamed column on read back
sales.to_csv("out/sales.csv")

# Correct
sales.to_csv("out/sales.csv", index=False)

**Treating `object` dtype as "the string dtype".** It is a catch-all,
not a string type. Operations on it pay full Python overhead. Convert
to `category` for low-cardinality columns and to `string` (the nullable
string dtype) for the rest.